In [ ]:
!pip install --upgrade -qqq uv
!uv pip install -qqq torch==2.8.0 torchvision bitsandbytes xformers==0.0.32.post2 \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 transformers==5.2.0

!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!pip install pillow datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.8 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 97ms
Prepared 2 packages in 756ms
Uninstalled 2 packages in 112ms
Installed 2 packages in 79ms
 - transformers==5.3.0
 + transformers==5.2.0
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 342ms
Prepared 4 packages in 18.76s
Installed 4 packages in 9ms
 + causal-conv1d==1.6.0
 + fla-core==0.4.2
 + flash-linear-attention==0.4.2
 + ninja==1.13.0


In [ ]:
import torch
from unsloth import FastVisionModel
from datasets import load_dataset
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Load with 4bit to save VRAM, but use Float32 for compute stability on T4
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
# Apply LoRA
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    random_state=3407,
)

Unsloth: Making `model.base_model.model.model.visual` require gradients


In [ ]:
# We must ensure NO BFloat16 tensors exist.
FastVisionModel.for_training(model)
model = model.to(torch.float32) # Force base to Float32


In [ ]:
# dataset
dataset = load_dataset("SimulaMet/Kvasir-VQA-x1", split="train").select(range(1000))
def formatting_prompts_func(examples):
    conversations = []
    for q, a, img in zip(examples["question"], examples["answer"], examples["image"]):
        conv = [
            {"role": "user", "content": [{"type": "text", "text": q}, {"type": "image", "image": img}]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]}
        ]
        conversations.append(conv)
    return {"messages": conversations}
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/143594 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15955 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    max_seq_length=1024,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=30,
        learning_rate=2e-4,
        fp16=False,
        bf16=False,
        optim="adamw_8bit",
        logging_steps=1,
        output_dir="outputs",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True}
    ),
)

Unsloth: Model does not have a default image size - using 512
Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 13,181,952 of 866,167,872 (1.52% trained)


Step,Training Loss
1,2.859499
2,2.624425
3,2.678454
4,2.254687
5,1.919302
6,2.050092
7,1.940706
8,1.584864
9,1.509977
10,1.156100


TrainOutput(global_step=30, training_loss=1.3883691211541493, metrics={'train_runtime': 724.469, 'train_samples_per_second': 0.331, 'train_steps_per_second': 0.041, 'total_flos': 222571309056000.0, 'train_loss': 1.3883691211541493, 'epoch': 0.24})

In [ ]:
# LoRA, light
model.save_pretrained("qwen3_5_kvasir_lora")
tokenizer.save_pretrained("qwen3_5_kvasir_lora")

['qwen3_5_kvasir_lora/processor_config.json']

In [ ]:
# Merge LoRA into the main model and save as 16-bit (Float16)
# This is much faster for future inference
model.save_pretrained_merged("qwen3_5_kvasir_merged", tokenizer, save_method = "merged_16bit")


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 1 files from cache to `qwen3_5_kvasir_merged`:   0%|          | 0/1 [00:00<?, ?it/s]
Unsloth: Copying 1 files from cache to `qwen3_5_kvasir_merged`: 100%|██████████| 1/1 [00:15<00:00, 15.97s/it]


Successfully copied all 1 files from cache to `qwen3_5_kvasir_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 7781.64it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:18<00:00, 18.54s/it]


Unsloth: Merge process complete. Saved to `/content/qwen3_5_kvasir_merged`


In [ ]:
from PIL import Image
from transformers import TextStreamer

from unsloth import FastVisionModel

# Load the merged model
model, tokenizer = FastVisionModel.from_pretrained(
    "qwen3_5_kvasir_merged",
    load_in_4bit = True,
)
FastVisionModel.for_inference(model)

test_sample = dataset[0]
# If 'dataset' was mapped, 'messages' contains the data.
# Let's pull the first user message:
image = test_sample["messages"][0]["content"][1]["image"]
question = test_sample["messages"][0]["content"][0]["text"]

FastVisionModel.for_inference(model)

instruction = [
    {"role": "user", "content": [
        #{"type": "text", "text": f"Analyze this endoscopy image: {question}"},
        {"type": "text", "text": f"{question}"},
        {"type": "image", "image": image}
    ]}
]

input_text = tokenizer.apply_chat_template(instruction, add_generation_prompt=True)
inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"\n--- Question: {question} ---")
print("--- Model Answer: ---")
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1, # Low temperature for medical accuracy
)

==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tokenizer you are loading from 'qwen3_5_kvasir_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



--- Question: Are there any abnormalities, polyps, or anatomical landmarks visible in the image? ---
--- Model Answer: ---
No evidence of polyps, anatomical landmarks, or abnormal findings observed<|im_end|>


In [ ]:
!pip install -qqq evaluate rouge_score sacrebleu tqdm

import torch
import gc
import evaluate
from unsloth import FastVisionModel
from datasets import load_dataset
from tqdm import tqdm
from PIL import Image


rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# Setup (first 10 rows)
eval_dataset = load_dataset("SimulaMet/Kvasir-VQA-x1", split="train").select(range(10))

def get_model_predictions(model_name, dataset):
    """Loads a model with higher sequence limits and runs inference."""
    print(f"\n--- Loading Model: {model_name} ---")

    # increase max_seq_length to 4096 to prevent image token truncation
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name,
        load_in_4bit=True,
        max_seq_length=4096,
    )
    FastVisionModel.for_inference(model)

    predictions = []
    references = []

    print(f"Running inference on {len(dataset)} samples...")
    for i in tqdm(range(len(dataset))):
        sample = dataset[i]
        image = sample["image"]
        question = sample["question"]
        reference = sample["answer"]

        instruction = [
            {"role": "user", "content": [
                {"type": "text", "text": f"{question}"},
                {"type": "image", "image": image}
            ]}
        ]

        input_text = tokenizer.apply_chat_template(instruction, add_generation_prompt=True)

        # Limit pixels to ensure the token count stays within the 4096 limit
        inputs = tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
            min_pixels=224*224,
            max_pixels=448*448,
        ).to("cuda")

        try:
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=64,
                    use_cache=True,
                    temperature=0.1,
                )

            # Extract only the assistant's response
            pred_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
            predictions.append(pred_text)
            references.append(reference)

        except Exception as e:
            print(f"Error on sample {i}: {e}")
            predictions.append("")
            references.append(reference)

    # Clean up VRAM
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, references


# Base model
base_preds, references = get_model_predictions("unsloth/Qwen3.5-0.8B", eval_dataset)

# Fine-tuned model
ft_preds, _ = get_model_predictions("qwen3_5_kvasir_merged", eval_dataset)

def calculate_metrics(preds, refs):
    valid_indices = [i for i, p in enumerate(preds) if p != ""]
    filtered_preds = [preds[i] for i in valid_indices]
    filtered_refs = [refs[i] for i in valid_indices]

    rouge_results = rouge.compute(predictions=filtered_preds, references=filtered_refs)
    bleu_results = bleu.compute(predictions=filtered_preds, references=filtered_refs)

    return {
        "ROUGE-1": rouge_results['rouge1'],
        "ROUGE-L": rouge_results['rougeL'],
        "BLEU": bleu_results['bleu']
    }

base_metrics = calculate_metrics(base_preds, references)
ft_metrics = calculate_metrics(ft_preds, references)

print("\n" + "="*50)
print(f"{'Metric':<15} | {'Base Model':<15} | {'Fine-Tuned':<15}")
print("-" * 50)
for key in base_metrics.keys():
    print(f"{key:<15} | {base_metrics[key]:.4f}{' ':<10} | {ft_metrics[key]:.4f}")
print("="*50)

print("\nSample Output")
print(f"Q: {eval_dataset[0]['question']}")
print(f"Base: {base_preds[0]}")
print(f"FT: {ft_preds[0]}")


--- Loading Model: unsloth/Qwen3.5-0.8B ---
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Running inference on 10 samples...


100%|██████████| 10/10 [01:01<00:00,  6.11s/it]



--- Loading Model: qwen3_5_kvasir_merged ---
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tokenizer you are loading from 'qwen3_5_kvasir_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Running inference on 10 samples...


100%|██████████| 10/10 [00:19<00:00,  1.91s/it]



Metric          | Base Model      | Fine-Tuned     
--------------------------------------------------
ROUGE-1         | 0.1510           | 0.4775
ROUGE-L         | 0.1036           | 0.4419
BLEU            | 0.0000           | 0.2374

--- Sample Output ---
Q: Are there any abnormalities, polyps, or anatomical landmarks visible in the image?
Base: Based on the visual evidence provided in the image, there are no visible abnormalities, polyps, or distinct anatomical landmarks that would suggest a specific diagnosis.

The image appears to be a close-up endoscopic view of the esophagus or stomach lining. The mucosa (the inner lining) appears relatively healthy with
FT: No evidence of polyps or anatomical landmarks observed


In [ ]:
!pip install -qqq nltk

In [ ]:
import evaluate
import numpy as np

# metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor") # Better for capturing synonyms and meaning

def calculate_metrics_improved(preds, refs):
    # Filter out empty predictions
    valid_indices = [i for i, p in enumerate(preds) if p.strip() != ""]
    filtered_preds = [preds[i] for i in valid_indices]
    filtered_refs = [refs[i] for i in valid_indices]

    if not filtered_preds:
        return {"Error": "No valid predictions to score"}

    # 1. ROUGE
    rouge_results = rouge.compute(predictions=filtered_preds, references=filtered_refs)

    # 2. BLEU with Smoothing
    # We use 'smooth=True' to give credit for partial matches (1-grams, 2-grams)
    bleu_results = bleu.compute(predictions=filtered_preds, references=filtered_refs, smooth=True)

    # 3. METEOR
    meteor_results = meteor.compute(predictions=filtered_preds, references=filtered_refs)

    return {
        "ROUGE-L": rouge_results['rougeL'],
        "BLEU (Smoothed)": bleu_results['bleu'],
        "METEOR": meteor_results['meteor']
    }

# Assuming you still have 'base_preds', 'ft_preds', and 'references' in memory:
base_metrics = calculate_metrics_improved(base_preds, references)
ft_metrics = calculate_metrics_improved(ft_preds, references)

print("\n" + "="*60)
print(f"{'Metric':<20} | {'Base Model':<15} | {'Fine-Tuned':<15}")
print("-" * 60)
for key in base_metrics.keys():
    print(f"{key:<20} | {base_metrics[key]:.4f}{' ':<10} | {ft_metrics[key]:.4f}")
print("="*60)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...



Metric               | Base Model      | Fine-Tuned     
------------------------------------------------------------
ROUGE-L              | 0.1036           | 0.4419
BLEU (Smoothed)      | 0.0084           | 0.2443
METEOR               | 0.1881           | 0.4380


II. Kvasir-Seg

In [ ]:
from datasets import load_dataset

ds = load_dataset("RGarrido03/kvasir-seg-augmented")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/822 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/412M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/51.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/51.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/600 [00:00<?, ? examples/s]

In [ ]:
from transformers import pipeline

pipe = pipeline("mask-generation", model="facebook/sam2-hiera-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/184M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/357 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("facebook/sam2-hiera-small")
model = AutoModel.from_pretrained("facebook/sam2-hiera-small")

Loading weights:   0%|          | 0/514 [00:00<?, ?it/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['image', 'mask'],
        num_rows: 4800
    })
    validation: Dataset({
        features: ['image', 'mask'],
        num_rows: 600
    })
    test: Dataset({
        features: ['image', 'mask'],
        num_rows: 600
    })
})

In [ ]:
import torch

pipe = pipeline("mask-generation", model="facebook/sam2-hiera-small")
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    pipe.set_image("img_kvasirseg.jpg")
    masks, _, _ = pipe.predict("colorectal polyps")


In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

  Cloning https://github.com/facebookresearch/segment-anything-2.git to /tmp/pip-req-build-n63qgvzk
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything-2.git /tmp/pip-req-build-n63qgvzk
  Resolved https://github.com/facebookresearch/segment-anything-2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.8 MB/s eta 0:00:00
  Created wheel for SAM-2: filename=sam_2-1.0-cp312-cp312-linux_x86_64.whl size=183669 sha256=92057c1ebbb6391cca72fa16e8cfaa11bc6d502fb442047252e46702436e61d1
  Stored in directory: /tmp/pip-ephem-wheel-cache-1vmaljvl/wheels/b0/5b/a1/c16dafb8c3c0046f1158e7fa7570559a72085df5beab759ba0
  Created wheel

In [ ]:
import os
import torch

# Instalare SAM 2
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

# Descărcare checkpoint-uri (folosim varianta 'tiny' pentru T4)
!wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt

  Cloning https://github.com/facebookresearch/segment-anything-2.git to /tmp/pip-req-build-53j79t9t
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything-2.git /tmp/pip-req-build-53j79t9t
  Resolved https://github.com/facebookresearch/segment-anything-2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
--2026-03-29 08:31:43--  https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.33, 13.249.182.39, 13.249.182.81, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 155906050 (149M) [application/vnd.snesdev-page-table]
Saving to: ‘checkpoints/sam2_hiera_tiny.pt’

sam2_hiera_tiny.pt  100%[=================

In [ ]:
import os
import torch

# Instalăm SAM 2 direct din sursă
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

# Descărcăm checkpoint-ul tiny pentru a rămâne în limitele memoriei T4 (la fel ca la Qwen)
!mkdir -p checkpoints
if not os.path.exists("checkpoints/sam2_hiera_tiny.pt"):
    !wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt

  Cloning https://github.com/facebookresearch/segment-anything-2.git to /tmp/pip-req-build-6zt9tpy7
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything-2.git /tmp/pip-req-build-6zt9tpy7
  Resolved https://github.com/facebookresearch/segment-anything-2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

checkpoint = "checkpoints/sam2_hiera_tiny.pt"
model_cfg = "sam2_hiera_t.yaml"

sam2_model = build_sam2(model_cfg, checkpoint, device="cuda")
predictor = SAM2ImagePredictor(sam2_model)